# SpyCloud Incident Triage Notebook
## Guided Investigation Workflow for SpyCloud Sentinel Incidents

This notebook provides a structured investigation workflow for triaging SpyCloud credential exposure incidents in Microsoft Sentinel.

**Sections:**
1. Setup & Configuration
2. Incident Context & Entity Extraction
3. SpyCloud Exposure Analysis
4. Entra ID / Sign-in Correlation
5. Device Investigation
6. Network & Firewall Correlation
7. O365 Activity Analysis
8. UEBA Behavioral Analysis
9. Remediation Status Check
10. Recommendations & Risk Score
11. Investigation Graph

## 1. Setup & Configuration

In [ ]:
# Install required packages
# !pip install msticpy pandas plotly networkx kqlmagic

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
import json

# Initialize msticpy
import msticpy as mp
mp.init_notebook(namespace=globals())

from msticpy.data import QueryProvider
from msticpy.vis import timeline as tl

# Connect to Sentinel workspace
qry_prov = QueryProvider("LogAnalytics")
qry_prov.connect(mp.WorkspaceConfig())

print("Connected to Sentinel workspace successfully.")

In [ ]:
# Set investigation parameters
INCIDENT_ID = input("Enter Sentinel Incident ID (or ARM ID): ")
LOOKBACK_DAYS = 30
print(f"Investigating incident: {INCIDENT_ID}")
print(f"Lookback period: {LOOKBACK_DAYS} days")

## 2. Incident Context & Entity Extraction

In [ ]:
# Get incident details
incident_query = f"""
SecurityIncident
| where IncidentNumber == '{INCIDENT_ID}' or IncidentName == '{INCIDENT_ID}'
| project IncidentNumber, Title, Severity, Status, Classification, Owner,
          CreatedTime, LastModifiedTime, ProviderName, AlertIds,
          AdditionalData, Labels
"""
incident_df = qry_prov.exec_query(incident_query)
if not incident_df.empty:
    display(incident_df)
else:
    print("Incident not found. Check the ID.")

# Extract entities
entity_query = f"""
SecurityIncident
| where IncidentNumber == '{INCIDENT_ID}' or IncidentName == '{INCIDENT_ID}'
| mv-expand Entity = todynamic(AdditionalData).entities
| project EntityType = tostring(Entity.Type), 
          EntityName = coalesce(tostring(Entity.Name), tostring(Entity.HostName), tostring(Entity.Address)),
          EntityDetails = Entity
"""
entities_df = qry_prov.exec_query(entity_query)
display(entities_df)

# Extract user entities for downstream queries
user_entities = entities_df[entities_df['EntityType'] == 'account']['EntityName'].tolist()
host_entities = entities_df[entities_df['EntityType'] == 'host']['EntityName'].tolist()
ip_entities = entities_df[entities_df['EntityType'] == 'ip']['EntityName'].tolist()
print(f"\nUsers: {user_entities}")
print(f"Hosts: {host_entities}")
print(f"IPs: {ip_entities}")

## 3. SpyCloud Exposure Analysis

In [ ]:
# Query SpyCloud exposure data for affected users
if user_entities:
    users_filter = "', '".join(user_entities)
    exposure_query = f"""
    SpyCloudBreachWatchlist_CL
    | where TimeGenerated > ago({LOOKBACK_DAYS}d)
    | where email in ('{users_filter}')
    | project TimeGenerated, email, severity, password_type, target_domain,
            infected_machine_id, user_hostname, country_code, source_id,
            sighting, spycloud_publish_date
    | order by TimeGenerated desc
    """
    exposure_df = qry_prov.exec_query(exposure_query)
    print(f"Found {len(exposure_df)} exposure records")
    display(exposure_df.head(20))
else:
    print("No user entities found in incident.")

In [ ]:
# Exposure timeline visualization
if not exposure_df.empty:
    timeline_df = exposure_df.groupby([pd.Grouper(key='TimeGenerated', freq='D'), 'severity']).size().reset_index(name='count')
    fig = px.bar(timeline_df, x='TimeGenerated', y='count', color='severity',
                 title='SpyCloud Exposure Timeline',
                 labels={'count': 'Exposures', 'TimeGenerated': 'Date'},
                 color_discrete_map={2: '#3498db', 5: '#f39c12', 20: '#e74c3c', 25: '#8e44ad'})
    fig.show()

    # Password type breakdown
    pwd_fig = px.pie(exposure_df, names='password_type', title='Password Type Distribution')
    pwd_fig.show()

    # Severity distribution
    sev_fig = px.histogram(exposure_df, x='severity', title='Severity Distribution',
                           nbins=4, color='severity')
    sev_fig.show()

In [ ]:
# Related breach catalog entries
if not exposure_df.empty:
    source_ids = exposure_df['source_id'].dropna().unique().tolist()
    if source_ids:
        source_filter = ', '.join([str(int(s)) for s in source_ids])
        catalog_query = f"""
        SpyCloudBreachCatalog_CL
        | where source_id in ({source_filter})
        | project short_title, breach_category, malware_family, confidence, 
                num_records, sensitive_source, spycloud_publish_date
        """
        catalog_df = qry_prov.exec_query(catalog_query)
        print("Related Breach Sources:")
        display(catalog_df)

## 4. Entra ID / Sign-in Correlation

In [ ]:
# Sign-in activity for affected users
if user_entities:
    signin_query = f"""
    SigninLogs
    | where TimeGenerated > ago({LOOKBACK_DAYS}d)
    | where UserPrincipalName in ('{users_filter}')
    | project TimeGenerated, UserPrincipalName, IPAddress, 
            ResultType, ResultDescription, AppDisplayName,
            LocationDetails, ConditionalAccessStatus, 
            AuthenticationRequirement, DeviceDetail
    | order by TimeGenerated desc
    """
    signin_df = qry_prov.exec_query(signin_query)
    print(f"Found {len(signin_df)} sign-in events")
    
    # Success vs failure breakdown
    signin_df['Status'] = signin_df['ResultType'].apply(lambda x: 'Success' if x == '0' else 'Failed')
    status_fig = px.histogram(signin_df, x='TimeGenerated', color='Status',
                              title='Sign-in Activity (Success vs Failed)',
                              barmode='stack')
    status_fig.show()
    
    # Geographic distribution
    if 'LocationDetails' in signin_df.columns:
        signin_df['Country'] = signin_df['LocationDetails'].apply(
            lambda x: x.get('countryOrRegion', 'Unknown') if isinstance(x, dict) else 'Unknown')
        geo_fig = px.pie(signin_df, names='Country', title='Sign-in Locations')
        geo_fig.show()

In [ ]:
# Risky sign-in events
if user_entities:
    risk_query = f"""
    AADUserRiskEvents
    | where TimeGenerated > ago({LOOKBACK_DAYS}d)
    | where UserPrincipalName in ('{users_filter}')
    | project TimeGenerated, UserPrincipalName, RiskLevel, RiskDetail,
            RiskEventType, IPAddress, Location
    | order by TimeGenerated desc
    """
    risk_df = qry_prov.exec_query(risk_query)
    print(f"Found {len(risk_df)} risk events")
    display(risk_df)

## 5. Device Investigation

In [ ]:
# SpyCloud device data
device_query = f"""
SpyCloudCompassDevices_CL
| where TimeGenerated > ago({LOOKBACK_DAYS}d)
| project TimeGenerated, user_hostname, user_os, ip_addresses,
        infected_time, application_count, log_id
| order by TimeGenerated desc
"""
device_df = qry_prov.exec_query(device_query)
print(f"Found {len(device_df)} infected device records")
display(device_df)

# MDE remediation logs
mde_log_query = f"""
Spycloud_MDE_Logs_CL
| where TimeGenerated > ago({LOOKBACK_DAYS}d)
| project TimeGenerated, DeviceName, DeviceId, Action, IsolationType,
        Tag, ActionStatus, Email, Severity
| order by TimeGenerated desc
"""
mde_log_df = qry_prov.exec_query(mde_log_query)
print(f"Found {len(mde_log_df)} MDE remediation actions")
display(mde_log_df)

## 6. Network & Firewall Correlation

In [ ]:
# Firewall events for SpyCloud IPs
if ip_entities:
    ips_filter = "', '".join(ip_entities)
    fw_query = f"""
    CommonSecurityLog
    | where TimeGenerated > ago(7d)
    | where SourceIP in ('{ips_filter}') or DestinationIP in ('{ips_filter}')
    | summarize EventCount = count(), Actions = make_set(DeviceAction) by 
            DeviceVendor, DeviceProduct, SourceIP, DestinationIP
    | order by EventCount desc
    """
    fw_df = qry_prov.exec_query(fw_query)
    print(f"Firewall events: {len(fw_df)}")
    display(fw_df)
else:
    print("No IP entities to check against firewalls.")

## 7. O365 Activity Analysis

In [ ]:
# O365 activity for exposed users
if user_entities:
    o365_query = f"""
    OfficeActivity
    | where TimeGenerated > ago(7d)
    | where UserId in ('{users_filter}')
    | summarize EventCount = count() by Operation, OfficeWorkload
    | order by EventCount desc
    | take 30
    """
    o365_df = qry_prov.exec_query(o365_query)
    print(f"O365 activity summary:")
    display(o365_df)
    
    # Check for suspicious mail rules
    rules_query = f"""
    OfficeActivity
    | where TimeGenerated > ago(30d)
    | where UserId in ('{users_filter}')
    | where Operation in ('New-InboxRule', 'Set-InboxRule', 'Set-Mailbox')
    | project TimeGenerated, UserId, Operation, Parameters, ClientIP
    """
    rules_df = qry_prov.exec_query(rules_query)
    if not rules_df.empty:
        print("WARNING: Suspicious mail rules detected!")
        display(rules_df)
    else:
        print("No suspicious mail rules found.")

## 8. UEBA Behavioral Analysis

In [ ]:
# UEBA anomalies for exposed users
if user_entities:
    ueba_query = f"""
    BehaviorAnalytics
    | where TimeGenerated > ago(7d)
    | where UserPrincipalName in ('{users_filter}')
    | project TimeGenerated, UserPrincipalName, ActivityType, 
            ActivityInsights, InvestigationPriority, SourceIPAddress
    | order by InvestigationPriority desc
    """
    ueba_df = qry_prov.exec_query(ueba_query)
    print(f"UEBA anomalies: {len(ueba_df)}")
    display(ueba_df)

## 9. Remediation Status Check

In [ ]:
# Check CA remediation logs
if user_entities:
    ca_query = f"""
    SpyCloud_ConditionalAccessLogs_CL
    | where TimeGenerated > ago({LOOKBACK_DAYS}d)
    | where Email in ('{users_filter}') or UserPrincipalName in ('{users_filter}')
    | project TimeGenerated, UserPrincipalName, Action, ActionStatus,
            PasswordReset, SessionsRevoked, AddedToCAGroup, Severity
    | order by TimeGenerated desc
    """
    ca_df = qry_prov.exec_query(ca_query)
    print(f"Remediation actions: {len(ca_df)}")
    display(ca_df)
    
    if ca_df.empty:
        print("\n*** WARNING: No remediation actions found for this user! ***")
        print("RECOMMENDED: Initiate password reset and session revocation immediately.")

## 10. Risk Score & Recommendations

In [ ]:
# Calculate composite risk score
risk_score = 0
risk_factors = []

if not exposure_df.empty:
    max_severity = exposure_df['severity'].max()
    if max_severity >= 25:
        risk_score += 40
        risk_factors.append(f"Critical severity exposure (severity={max_severity}): +40")
    elif max_severity >= 20:
        risk_score += 30
        risk_factors.append(f"High severity exposure (severity={max_severity}): +30")
    elif max_severity >= 5:
        risk_score += 15
        risk_factors.append(f"Medium severity exposure (severity={max_severity}): +15")
    
    if 'plaintext' in exposure_df['password_type'].values:
        risk_score += 20
        risk_factors.append("Plaintext password exposed: +20")
    
    exposure_count = len(exposure_df)
    if exposure_count > 10:
        risk_score += 10
        risk_factors.append(f"Multiple exposures ({exposure_count}): +10")

if 'signin_df' in dir() and not signin_df.empty:
    successful_after = signin_df[signin_df['Status'] == 'Success']
    if len(successful_after) > 0:
        risk_score += 15
        risk_factors.append(f"Successful sign-ins detected ({len(successful_after)}): +15")

if 'risk_df' in dir() and not risk_df.empty:
    risk_score += 15
    risk_factors.append(f"Identity Protection risk events ({len(risk_df)}): +15")

if 'ca_df' in dir() and ca_df.empty:
    risk_score += 10
    risk_factors.append("No remediation actions taken: +10")

print(f"\n{'='*60}")
print(f"COMPOSITE RISK SCORE: {risk_score}/100")
print(f"{'='*60}")
for factor in risk_factors:
    print(f"  {factor}")

print(f"\n{'='*60}")
print("RECOMMENDED ACTIONS:")
print(f"{'='*60}")
if risk_score >= 70:
    print("  [CRITICAL] Immediate password reset required")
    print("  [CRITICAL] Revoke all sessions and tokens")
    print("  [CRITICAL] Force MFA re-registration")
    print("  [CRITICAL] Add to SpyCloud-Blocked security group")
    print("  [HIGH] Isolate infected devices")
    print("  [HIGH] Block compromised IPs in firewall")
elif risk_score >= 40:
    print("  [HIGH] Reset password within 24 hours")
    print("  [HIGH] Revoke sessions")
    print("  [MEDIUM] Add to SpyCloud-Restricted security group")
    print("  [MEDIUM] Investigate infected devices")
else:
    print("  [MEDIUM] Schedule password reset")
    print("  [LOW] Add to SpyCloud-Monitored security group")
    print("  [LOW] Review sign-in patterns")

## 11. Investigation Graph

In [ ]:
import networkx as nx
import plotly.graph_objects as go

G = nx.Graph()

# Add user nodes
for user in user_entities:
    G.add_node(user, type='user', color='#e74c3c')

# Add exposure relationships
if not exposure_df.empty:
    for _, row in exposure_df.iterrows():
        if pd.notna(row.get('target_domain')):
            G.add_node(row['target_domain'], type='domain', color='#3498db')
            G.add_edge(row['email'], row['target_domain'], relation='exposed_on')
        if pd.notna(row.get('infected_machine_id')):
            G.add_node(str(row['infected_machine_id'])[:12], type='device', color='#f39c12')
            G.add_edge(row['email'], str(row['infected_machine_id'])[:12], relation='infected_device')

# Add IP relationships
for ip in ip_entities:
    G.add_node(ip, type='ip', color='#2ecc71')
    for user in user_entities:
        G.add_edge(user, ip, relation='connected_from')

# Layout and visualization
pos = nx.spring_layout(G, k=2, iterations=50)

edge_x, edge_y = [], []
for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

node_x = [pos[n][0] for n in G.nodes()]
node_y = [pos[n][1] for n in G.nodes()]
node_colors = [G.nodes[n].get('color', '#95a5a6') for n in G.nodes()]
node_labels = list(G.nodes())

fig = go.Figure()
fig.add_trace(go.Scatter(x=edge_x, y=edge_y, mode='lines', line=dict(width=1, color='#bdc3c7')))
fig.add_trace(go.Scatter(x=node_x, y=node_y, mode='markers+text', text=node_labels,
                         textposition='top center', marker=dict(size=15, color=node_colors)))
fig.update_layout(title='Investigation Entity Graph', showlegend=False, 
                  xaxis=dict(visible=False), yaxis=dict(visible=False))
fig.show()
print(f"Graph: {len(G.nodes())} entities, {len(G.edges())} relationships")